# Fine-tuning a Medical Q&A Assistant with QLoRA (Mistral-7B-Instruct-v0.2)**Runtime required: GPU.** In Colab: `Runtime > Change runtime type > T4 GPU` (or better).This notebook drives the scripts in `src/` end-to-end:1. `prepare_dataset.py` — download + clean the medical Q&A dataset, format it for Mistral's instruct template2. `finetune.py` — QLoRA fine-tuning (4-bit base model + LoRA adapter)3. `evaluate.py` — base vs. fine-tuned comparison on held-out questions (ROUGE-L + qualitative examples)4. `inference.py` — ask the fine-tuned model a question interactivelySee `docs/theory.md` for the quantization/LoRA/QLoRA math behind steps 1-2, and the top-level `README.md` for the full project write-up.**Note on dependencies:** `requirements.txt` is pinned to current package releases rather than the mid-2024 versions in the original write-up, since Colab now runs Python 3.13 and the older pins (in particular an old `tokenizers` release with no Python 3.13 wheel) fail to build. The QLoRA implementation itself — 4-bit `BitsAndBytesConfig`, `LoraConfig`, `SFTTrainer` — is unchanged.**Disclaimer:** this is a fine-tuning skills demo on a small open medical Q&A dataset. It is *not* a clinically validated tool and must never be used for real medical decisions.

## 0. Get the project onto this Colab instance

In [ ]:
!git clone https://github.com/Kritikaagra/medical-qa-assistant.git
%cd medical-qa-assistant

## 1. Install dependenciesThis reinstalls several packages Colab ships by default (e.g. `transformers`, `accelerate`). **If a "Restart session" button/banner appears after this cell finishes, click it**, then continue running from the next cell — Colab needs a fresh process to pick up the upgraded packages. You do not need to redo the clone.

In [ ]:
!pip3 install -q -U -r requirements.txt

In [ ]:
# Sanity check: confirm the GPU is visible and the key packages imported cleanly.
import torch
print("Python:", __import__("sys").version)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU detected — go to Runtime > Change runtime type and select a GPU.")

import transformers, accelerate, peft, trl, bitsandbytes
print("transformers", transformers.__version__)
print("accelerate ", accelerate.__version__)
print("peft       ", peft.__version__)
print("trl        ", trl.__version__)
print("bitsandbytes", bitsandbytes.__version__)

## 2. Hugging Face authentication`mistralai/Mistral-7B-Instruct-v0.2` is ungated, so a token isn't strictly required to download it — but set one anyway to avoid rate limits, and because you'll want it if you swap in a gated model (e.g. Llama 2, Gemma) later.Add your token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar), then run:

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

## 3. Prepare the dataset (no GPU needed for this step)

In [ ]:
!python src/prepare_dataset.py

## 4. QLoRA fine-tuningOn a free T4 with the default `config.py` settings (~3000 training examples, 1 epoch, batch size 4 x grad-accum 4), this takes roughly 20-40 minutes. Loss should be logged every 10 steps.

In [ ]:
!python src/finetune.py

## 5. Evaluate: base model vs. fine-tuned adapterGenerates answers from both the untouched base model and the fine-tuned adapter on the same held-out questions, scores them with ROUGE-L against the reference answers, and writes `results/eval_results.md` + `results/eval_raw.json`.

In [ ]:
!python src/evaluate.py

In [ ]:
from IPython.display import Markdown, display

with open("results/eval_results.md") as f:
    display(Markdown(f.read()))

## 6. Try it yourself

In [ ]:
import sys
sys.path.insert(0, "src")

import inference

model, tokenizer = inference.load_model_and_tokenizer()
print(inference.ask(model, tokenizer, "What are the early symptoms of anemia?"))

## Next steps- Try a higher LoRA rank (`LORA_R` in `src/config.py`) if you have more GPU time, and compare ROUGE-L / qualitative quality against this run.- Swap in a different open base model (e.g. `google/gemma-2b-it`, gated — needs license acceptance on Hugging Face) by changing `BASE_MODEL_ID` and adjusting `LORA_TARGET_MODULES` for that architecture.- See `docs/theory.md` for the quantization/LoRA math this pipeline relies on.